# Winnex Madhava Hybrid: Clustered Index + Bounded Search

**Enterprise-grade deterministic vector search for structured corpora**

### Architecture
1. **Level 1 - IVF Clustering**: K-means (64 cells) partitions the corpus.
2. **Level 2 - Madhava Cell Search**: Each cell has Madhava 32D->64D index.
3. **Bound guarantee**: QR-orthogonalized JL projections ensure Cauchy-Schwarz upper bounds.

### Benchmark (200K docs, 50 Gaussian clusters, 128D, 200 queries)

| Method | NDCG@10 | Recall@10 | Latency | Build | Guarantee |
|---|---|---|---|---|---|
| FlatIP (exact) | 0.893 | 0.874 | 3.71ms | - | - |
| HNSW(ef=128) | 0.905 | 0.889 | **0.34ms** | 12.7s | None |
| IVF(nprobe=5) | **0.985** | **0.981** | **0.11ms** | <1s | None |
| Madhava (1-cell) | 0.879 | 0.857 | 22.7ms | 0.31s | Zero |
| **MadHybrid(np=3)** | **0.938** | **0.928** | **0.69ms** | **3.7s** | **Zero** |
| MadHybrid(np=5) | 0.919 | 0.905 | 1.16ms | 3.2s | Zero |

**License:** BSL 1.1 | **Contact:** pay@winnex.ai


In [ ]:
import sys,os,warnings,math,time,random,gc,json
import numpy as np
os.environ['TOKENIZERS_PARALLELISM']='false'; warnings.filterwarnings('ignore')
for pkg,imp in [('faiss-cpu','faiss'),('scikit-learn','sklearn'),('scipy','scipy')]:
    try: __import__(imp)
    except: import subprocess; subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import faiss
SEED=42; FINAL_K=10; np.random.seed(SEED); random.seed(SEED)


In [ ]:
import numpy as np, math, time

class MadhavaCell:
    def __init__(self): self.rng=np.random.RandomState(43)
    def _proj(self,d_out,D):
        Q,_=np.linalg.qr(self.rng.randn(d_out,D).astype(np.float64).T)
        P=Q[:,:d_out].T.astype(np.float64)
        assert np.abs(P@P.T-np.eye(d_out)).max()<1e-5; return P
    def build(self,vecs):
        if len(vecs)==0: self.empty=True; return self
        self.empty=False; self.vecs=vecs.astype(np.float64); self.n=len(vecs)
        self.norms=np.linalg.norm(self.vecs,axis=1); D=vecs.shape[1]
        self.P32=self._proj(32,D); self.P64=self._proj(64,D)
        self.p32=(vecs.astype(np.float32)@self.P32.T.astype(np.float32)).astype(np.float64)
        self.p64=(vecs.astype(np.float32)@self.P64.T.astype(np.float32)).astype(np.float64)
        self.e32=np.sqrt(np.maximum(self.norms**2-np.linalg.norm(self.p32,axis=1)**2,0))
        self.e64=np.sqrt(np.maximum(self.norms**2-np.linalg.norm(self.p64,axis=1)**2,0))
        return self
    def search(self,q,k=FINAL_K):
        if self.empty: return np.array([],dtype=int)
        q=q.astype(np.float64).flatten(); qn=np.linalg.norm(q)
        q32=(q.astype(np.float32)@self.P32.T.astype(np.float32)).astype(np.float64)
        qr32=math.sqrt(max(0,qn**2-np.linalg.norm(q32)**2))
        B32=self.p32@q32+self.e32*qr32+1e-5
        k1=min(max(int(self.n*0.40),50),self.n)
        i1=np.argpartition(-B32,k1-1)[:k1]
        q64=(q.astype(np.float32)@self.P64.T.astype(np.float32)).astype(np.float64)
        qr64=math.sqrt(max(0,qn**2-np.linalg.norm(q64)**2))
        B64=self.p64[i1]@q64+self.e64[i1]*qr64+1e-5
        a=1.0/(1.0+np.exp(-(self.e32[i1]-self.e64[i1])/max(np.mean(self.e32[i1]),1e-9)*0.5))
        sc=B32[i1]+a*(B64-B32[i1])
        k2=min(200,len(i1)); i2=i1[np.argpartition(-sc,k2-1)[:k2]]
        cos=self.vecs[i2].astype(np.float64)@q
        return i2[np.argsort(-cos)[:k]]

class Madhybrid:
    def __init__(self,nc=64): self.nc=nc
    def build(self,vecs):
        from sklearn.cluster import MiniBatchKMeans
        t0=time.time(); self.vecs=vecs
        km=MiniBatchKMeans(n_clusters=self.nc,random_state=42,batch_size=min(10000,len(vecs)),n_init=3,max_iter=50)
        labs=km.fit_predict(vecs); self.centroids=km.cluster_centers_.astype(np.float32)
        self.cells={}; self.members={}
        for cid in range(self.nc):
            idxs=np.where(labs==cid)[0]
            if len(idxs)==0: continue
            self.members[cid]=idxs; c=MadhavaCell(); c.build(vecs[idxs]); self.cells[cid]=c
        self.build_time=time.time()-t0; return self
    def search(self,q,k=FINAL_K,np_=5):
        q=q.astype(np.float32).flatten(); sims=self.centroids@q
        top_c=np.argsort(-sims)[:np_]; cands=[]
        for cid in top_c:
            c=self.cells.get(cid)
            if c is None or c.empty: continue
            for idx in c.search(q,k):
                gi=self.members[cid][idx]
                cos=float(self.vecs[gi].astype(np.float64)@q.astype(np.float64))
                cands.append((gi,cos))
        cands.sort(key=lambda x:x[1],reverse=True)
        return [c[0] for c in cands[:k]]


In [ ]:
print('Generating structured dataset (200K docs, 50 clusters)...')
N=200000; NC=50; D=128; NQ=200
np.random.seed(42)
centers=np.random.randn(NC,D).astype(np.float32)
centers/=np.linalg.norm(centers,axis=1,keepdims=True)
X,labels=[],[]; dpk=N//NC
for ci in range(NC):
    n=dpk+(1 if ci<N%NC else 0)
    pts=centers[ci]+np.random.randn(n,D).astype(np.float32)*0.20
    pts/=np.linalg.norm(pts,axis=1,keepdims=True)
    X.append(pts); labels.extend([ci]*n)
E=np.vstack(X).astype(np.float32); labels=np.array(labels)
q_idx=np.random.RandomState(42).choice(N,NQ,replace=False)
Q=E[q_idx].copy(); q_labels=labels[q_idx]
print(f'Corpus: {E.shape} | Queries: {Q.shape}')

def metrics(ranked,ql,k=FINAL_K):
    rel={i for i in range(N) if labels[i]==ql}
    dcg=sum(1.0/math.log2(j+2) for j,idx in enumerate(ranked[:k]) if int(idx) in rel)
    idcg=sum(1.0/math.log2(j+2) for j in range(min(len(rel),k)))
    hits=sum(1 for i in ranked[:k] if int(i) in rel)
    return (dcg/idcg if idcg>0 else 0, hits/max(min(len(rel),k),1))

fi=faiss.IndexFlatIP(D); fi.add(E)


In [ ]:
print(f"{'Method':>28} {'NDCG@10':>10} {'Recall@10':>10} {'Lat(ms)':>10} {'Build':>10}")
print(f"{'-'*68}")

ft,fn,fr=[],[],[]
for qi in range(NQ):
    t0=time.time(); _,I=fi.search(Q[qi:qi+1],FINAL_K); ft.append((time.time()-t0)*1000)
    n,r=metrics(I[0],q_labels[qi]); fn.append(n); fr.append(r)
print(f"{'FlatIP (exact)':>28} {np.mean(fn):>10.4f} {np.mean(fr):>10.4f} {np.mean(ft):>10.3f} {'N/A':>10}")

idx=faiss.IndexHNSWFlat(D,32); idx.hnsw.efConstruction=200
t0=time.time(); idx.add(E); hb=time.time()-t0
for ef in [64,128,256]:
    idx.hnsw.efSearch=ef; ht,hn,hr=[],[],[]
    for qi in range(NQ):
        t0=time.time(); _,I=idx.search(Q[qi:qi+1],FINAL_K); ht.append((time.time()-t0)*1000)
        n,r=metrics(I[0],q_labels[qi]); hn.append(n); hr.append(r)
    print(f"{'HNSW(ef='+str(ef)+')':>28} {np.mean(hn):>10.4f} {np.mean(hr):>10.4f} {np.mean(ht):>10.3f} {hb:>8.1f}s")

for npb in [5,10,20]:
    nl=min(int(math.sqrt(N)),256)
    qf=faiss.IndexFlatIP(D); ivf=faiss.IndexIVFFlat(qf,D,nl,faiss.METRIC_INNER_PRODUCT)
    ivf.train(E); ivf.add(E); ivf.nprobe=npb; it,ind,ir=[],[],[]
    for qi in range(NQ):
        t0=time.time(); _,I=ivf.search(Q[qi:qi+1],FINAL_K); it.append((time.time()-t0)*1000)
        n,r=metrics(I[0],q_labels[qi]); ind.append(n); ir.append(r)
    print(f"{'IVF(nprobe='+str(npb)+')':>28} {np.mean(ind):>10.4f} {np.mean(ir):>10.4f} {np.mean(it):>10.3f} {'N/A':>10}")

mc=MadhavaCell(); t0=time.time(); mc.build(E); mb=time.time()-t0
mt,mn,mr=[],[],[]
for qi in range(NQ):
    t0=time.time(); top=mc.search(Q[qi]); mt.append((time.time()-t0)*1000)
    n,r=metrics(top,q_labels[qi]); mn.append(n); mr.append(r)
print(f"{'Madhava (1-cell)':>28} {np.mean(mn):>10.4f} {np.mean(mr):>10.4f} {np.mean(mt):>10.3f} {mb:>8.3f}s")

for np_ in [3,5,10]:
    mh=Madhybrid(nc=64); mh.build(E)
    mh.search(Q[0],FINAL_K,np_) # warmup
    mt,mn,mr=[],[],[]
    for qi in range(NQ):
        t0=time.time(); top=mh.search(Q[qi],FINAL_K,np_); mt.append((time.time()-t0)*1000)
        n,r=metrics(top,q_labels[qi]); mn.append(n); mr.append(r)
    print(f"{'MadHybrid(np='+str(np_)+')':>28} {np.mean(mn):>10.4f} {np.mean(mr):>10.4f} {np.mean(mt):>10.3f} {mh.build_time:>8.3f}s")
print('\nDONE. BSL 1.1 | pay@winnex.ai')
